In [6]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [7]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"
BATCH_SIZE = 32
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
target_col = "Output"

In [8]:
# Feature sets
feature_sets = [
    {
        "name": "LSTM Classifier (Explainability Features)",
        "features": ['TCD', 'PC', 'CN0', 'EC', 'PRN', 'LC', 'DO', 'PD', 'CP', 'PIP']
    },
    {
        "name": "LSTM Classifier (XGBoost Features)",
        "features": ['TOW', 'PD', 'PRN', 'RX', 'TCD', 'DO', 'CP', 'CN0', 'PC', 'EC']
    }
]

In [9]:
# Load the dataset
try:
    df = pd.read_excel(DATA_PATH)
except FileNotFoundError:
    print(f"Error: The file '{DATA_PATH}' was not found.")
    exit()

# Label encode target if needed
y = df[target_col]
if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)
else:
    y = y.values

In [11]:
def run_lstm_for_feature_set(feature_set_name, feature_cols, y, df):
    print(f"Training model: {feature_set_name}")
    X = df[feature_cols]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Prepare data for LSTM: (samples, sequence_length, input_size)
    INPUT_SIZE = 1
    SEQUENCE_LENGTH = X_train_scaled.shape[1]
    X_train_lstm = X_train_scaled.astype('float32').reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
    X_test_lstm = X_test_scaled.astype('float32').reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
    y_train_np = y_train.astype('int64')
    y_test_np = y_test.astype('int64')

    # Torch tensors
    X_train_tensor = torch.tensor(X_train_lstm)
    y_train_tensor = torch.tensor(y_train_np)
    X_test_tensor = torch.tensor(X_test_lstm)
    y_test_tensor = torch.tensor(y_test_np)

    # DataLoader
    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    test_data = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

    # Model definition
    class LSTMClassifier(nn.Module):
        def __init__(self, input_size, hidden_size, num_classes):
            super(LSTMClassifier, self).__init__()
            self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
            self.fc = nn.Linear(hidden_size, num_classes)

        def forward(self, x):
            out, _ = self.lstm(x)
            out = out[:, -1, :]
            out = self.fc(out)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(RANDOM_STATE)
        torch.cuda.manual_seed_all(RANDOM_STATE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Training
    start_time = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        if (epoch+1) % 10 == 0 or epoch == 0 or epoch == NUM_EPOCHS-1:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    train_time = time.time() - start_time

    # Evaluation
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, digits=4))
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    print("Accuracy:", accuracy * 100)
    print("F1 Score:", f1 * 100)

    # Save results for this model
    results = [{
        "model": feature_set_name,
        "dataset": os.path.basename(DATA_PATH),
        "f1score": f1,
        "accuracy": accuracy,
        "train_time_sec": train_time,
        "parameters": str(model.state_dict().keys()),
        "num_epochs": NUM_EPOCHS
    }]
    results_df = pd.DataFrame(results)
    file_exists = os.path.isfile(RESULTS_CSV)
    results_df.to_csv(
        RESULTS_CSV,
        mode='a',
        header=not file_exists,
        index=False
    )
    print(f"Results appended to {RESULTS_CSV}\n")

# Run for both feature sets
for fs in feature_sets:
    run_lstm_for_feature_set(fs["name"], fs["features"],y, df)

Training model: LSTM Classifier (Explainability Features)
Epoch 1/100, Loss: 0.6938, Accuracy: 0.7813
Epoch 10/100, Loss: 0.1842, Accuracy: 0.9059
Epoch 20/100, Loss: 0.1482, Accuracy: 0.9193
Epoch 30/100, Loss: 0.1343, Accuracy: 0.9253
Epoch 40/100, Loss: 0.1244, Accuracy: 0.9283
Epoch 50/100, Loss: 0.1226, Accuracy: 0.9304
Epoch 60/100, Loss: 0.1147, Accuracy: 0.9339
Epoch 70/100, Loss: 0.1120, Accuracy: 0.9348
Epoch 80/100, Loss: 0.1086, Accuracy: 0.9369
Epoch 90/100, Loss: 0.1052, Accuracy: 0.9394
Epoch 100/100, Loss: 0.1036, Accuracy: 0.9408

Classification Report:
              precision    recall  f1-score   support

           0     0.9530    0.9709    0.9619    318260
           1     0.7874    0.7972    0.7923     29166
           2     0.7693    0.6387    0.6979     35386
           3     0.9253    0.9136    0.9194     25612

    accuracy                         0.9261    408424
   macro avg     0.8587    0.8301    0.8429    408424
weighted avg     0.9235    0.9261    0.9242